# Sse

> SSE parsing helpers.

In [ ]:
#| default_exp sse

## Imports

In [ ]:
#| export
from dataclasses import dataclass
import httpx,json
from fastcore.utils import *

from fastllm.errors import *

In [ ]:
#| hide
from cachy.core import enable_cachy
from fastcore.test import *

In [ ]:
enable_cachy()

## Purpose And Design

`sse.py` handles Server-Sent Events parsing in a provider-neutral way.

### Why this module exists

Streaming endpoints from different providers still rely on SSE framing rules. A dedicated parser keeps framing concerns separate from provider-specific event semantics.

### Architectural fit

- Upstream: line streams from HTTP responses.
- Downstream: `transport.stream_sse(_json)` and then provider normalizers.

### Key design choices

- Preserve SSE semantics (`event`, `data`, `id`, `retry`).
- Accept incremental/partial line sources robustly.
- Keep parser minimal and dependency-free.

### Reader outcome

After this notebook, you can reason about where a stream framing issue should be fixed (here) versus where provider event interpretation should be fixed (`normalize.py`).

### `SSEvent`

A parsed SSE event.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Core fields: `event: Optional[str]`, `data: str`, `id: Optional[str]`, `retry: Optional[int]`
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `transport`.
- Module downstream: `transport`.

In [ ]:
#| export
@dataclass(frozen=True)
class SSEvent:
    "A parsed SSE event."
    event: str
    data: str
    id: str = None
    retry: int = None

Similar to what've done in 00_errors dialog, please refer to the official docs regarding SSE events

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me look up the official SSE spec.



<details class='tool-usage-details'>
<summary><code>flash_searches(prs=&quot;[&#x27;SSE server-sent events official spec f…&quot;)→&quot;[&#x27;The Server-Sent Events (SSE) specifica…&quot;</code></summary>

```json
{
  "id": "toolu_01AEStHoMQcQekKgtDdu5du2",
  "call": {
    "function": "flash_searches",
    "arguments": {
      "prs": "['SSE server-sent events official spec fields event data id retry', 'MDN server-sent events format specification']"
    }
  },
  "result": "<TRUNCATED>\u2026er-Sent Events (SSE) specification defines a set of fields that structure the messages exchanged between a server and a client over a persistent HTTP connection. These fields allow for categorized events, data payloads, event identification, and control over reconnection attempts.\\n\\nThe primary fields used in SSE are:\\n\\n*   **`event`**: This field specifies the type of the event. If provided, the client can use `addEventListener` to listen for specific event names. If omitted, the event is treated as a generic \"message\" and handled by the `onmessage` event handler. [baeldung.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHykXHrPSxeE9Xc-Ml5awLIjVV97jNoXMmJwnC5Cka9wkOjArkYZ_oSysuf-hMsfyMk0tJhgx59P9TDUSpu876EtQ0et9xzm2iTKJfPH06V3IMhd3jreraDGoCFmsD2scp_EFVs9dw=)\\n*   **`data`**: This field carries the actual message content sent by the server. Multiple `data` lines within a single event are concatenated, with newline characters separating them. [baeldung.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHykXHrPSxeE9Xc-Ml5awLIjVV97jNoXMmJwnC5Cka9wkOjArkYZ_oSysuf-hMsfyMk0tJhgx59P9TDUSpu876EtQ0et9xzm2iTKJfPH06V3IMhd3jreraDGoCFmsD2scp_EFVs9dw=)\\n*   **`id`**: This field serves as a unique identifier for an event. When the connection is re-established after being lost, the client sends the last received `id` in the `Last-Event-ID` header to the server. This allows the server to potentially resume sending events from where they left off, preventing data loss. [baeldung.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHykXHrPSxeE9Xc-Ml5awLIjVV97jNoXMmJwnC5Cka9wkOjArkYZ_oSysuf-hMsfyMk0tJhgx59P9TDUSpu876EtQ0et9xzm2iTKJfPH06V3IMhd3jreraDGoCFmsD2scp_EFVs9dw=)\\n*   **`retry`**: This field specifies the reconnection time in milliseconds. If the connection to the server is interrupted, the client will wait for this duration before attempting to reconnect. [baeldung.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHykXHrPSxeE9Xc-Ml5awLIjVV97jNoXMmJwnC5Cka9wkOjArkYZ_oSysuf-hMsfyMk0tJhgx59P9TDUSpu876EtQ0et9xzm2iTKJfPH06V3IMhd3jreraDGoCFmsD2scp_EFVs9dw=)\\n\\nThese fields are sent as plain text, UTF-8 encoded, and messages are delimited by double newline characters (`\\\\n\\\\n`). [mozilla.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHMHVWHt_VsutB9JulbywPlLjdvkdPi091r_T5ZtjlgpfhNuJIpLlEALtE4hjXNertpF_cfnYFVD4Fgs3yfURYY9-Q0oifk4gGOt32K9u8ELvIyVz4jSSk_zzCnwu7ZiGFG3SDJOfgqAGyNoDtuxuyvUXPND-4ry9VhD03Q8og320aZFOLodURTEpsmEvcu7e6-2796EydG) Comments can also be included in the stream, starting with a colon (`:`), and are ignored by the client. [mozilla.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHMHVWHt_VsutB9JulbywPlLjdvkdPi091r_T5ZtjlgpfhNuJIpLlEALtE4hjXNertpF_cfnYFVD4Fgs3yfURYY9-Q0oifk4gGOt32K9u8ELvIyVz4jSSk_zzCnwu7ZiGFG3SDJOfgqAGyNoDtuxuyvUXPND-4ry9VhD03Q8og320aZFOLodURTEpsmEvcu7e6-2796EydG)', 'The Server-Sent Events (SSE) format specification defines a simple text-based protocol for servers to push data to clients over HTTP. The connection is established using the `EventSource` interface on the client-side, and the server responds with the `text/event-stream` MIME type [wikipedia.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEFXg_jsJ9I7HIibwcyh0KKkL3kr829UH5tSQuW97zwxlDIEMAfIPwVpIEO7_QLoO-IND_Ib9YLUEEf2LUkplfT5Rot9l8Lk7QdNqPelVVoly4j3TEQbi_u_LrmXs515lwicqfpijMYzLhz-A==).\\n\\nKey aspects of the SSE format include:\\n\\n*   **UTF-8 Encoding:** The entire event stream must be encoded using UTF-8 [mozilla.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHagwJFXZfQ3SsP8GUf8y5sAO3v_RWhpxYUqxkASnC-XTCi03RFNQBB2a_rpA3SXdGZ7t_Zfx-ErRVxEK7VNVXzr8HZ35pecL6jbhEYLvanqUkZ80ReBrEFU-DWiwQgTPBvhmu4Wrjy05ebH4N2HtbdHRkH99qlQAUIj7M83kz37z3ERGSpbniU7JMNhBRSF-wROBXT8aau).\\n*   **Message Delimitation:** Messages within the event stream are separated by a pair of newline characters (`\\\\n\\\\n`) [mozilla.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHagwJFXZfQ3SsP8GUf8y5sAO3v_RWhpxYUqxkASnC-XTCi03RFNQBB2a_rpA3SXdGZ7t_Zfx-ErRVxEK7VNVXzr8HZ35pecL6jbhEYLvanqUkZ80ReBrEFU-DWiwQgTPBvhmu4Wrjy05ebH4N2HtbdHRkH99qlQAUIj7M83kz37z3ERGSpbniU7JMNhBRSF-wROBXT8aau).\\n*   **Fields:** Each message consists of one or more lines, each representing a field. The common fields are:\\n    *   `event`: A string that identifies the type of event. If specified, the client-side code should use `addEventListener()` to listen for named events. If omitted, a generic `message` event is fired [mozilla.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHagwJFXZfQ3SsP8GUf8y5sAO3v_RWhpxYUqxkASnC-XTCi03RFNQBB2a_rpA3SXdGZ7t_Zfx-ErRVxEK7VNVXzr8HZ35pecL6jbhEYLvanqUkZ80ReBrEFU-DWiwQgTPBvhmu4Wrjy05ebH4N2HtbdHRkH99qlQAUIj7M83kz37z3ERGSpbniU7JMNhBRSF-wROBXT8aau).\\n    *   `data`: The actual data for the message. If multiple conse\u2026</TRUNCATED>"
}
```

</details>

Here's a concise reference from the official specs:

The SSE protocol is defined in the [WHATWG HTML Living Standard §9.2](https://html.spec.whatwg.org/multipage/server-sent-events.html) (which supersedes the older [W3C EventSource spec](https://www.w3.org/TR/2012/WD-eventsource-20120426/)). The [MDN reference](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events) is also excellent.

Key points relevant to `SSEvent`:

- [*](https://www.aklivity.io/post/a-primer-on-server-sent-events-sse "A Primer on Server-Sent Events (SSE) | Aklivity Blog") Messages are text-based, UTF-8 encoded and have the following optional fields: `event:`, `data:`, `id:` and `retry:`.
- [*](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events "Using server-sent events - Web APIs | MDN") Each notification is sent as a block of text terminated by a pair of newlines.
- [*](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events "Using server-sent events - Web APIs | MDN") Messages sent from the server that don't have an `event` field are received as `message` events.
- [*](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events "Using server-sent events - Web APIs | MDN") `id` sets the EventSource object's last event ID value — [*](https://medium.com/@moali314/server-sent-events-a-comprehensive-guide-e4b15d147576 "Server-Sent Events: A Comprehensive Guide | by Mohamed-Ali | Medium") SSE's event ID mechanism enables reliable message delivery by allowing clients to resume streams from where they left off. When a client reconnects, it sends the `Last-Event-ID` header with the ID of the last event it received. The server can then send any events that occurred during the disconnection.
- [*](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events "Using server-sent events - Web APIs | MDN") `retry` is the reconnection time. If the connection to the server is lost, the browser will wait for the specified time before attempting to reconnect. This must be an integer, specifying the reconnection time in milliseconds. If a non-integer value is specified, the field is ignored.
- [*](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events "Using server-sent events - Web APIs | MDN") All other field names are ignored. If a line doesn't contain a colon, the entire line is treated as the field name with an empty value string.

So your `SSEvent` fields (`event`, `data`, `id`, `retry`) map exactly to the four SSE fields. Want me to suggest adding a note referencing the spec in the Purpose And Design or SSEvent docstring?

<details class='token-usage-details'><summary>$0.4464</summary>

`total=69,007 | in=68,241 | out=766 | cached=36.3% | cache_new=38,945 | searches=1 | $0.4464`

</details>


Now look at how different providers return SSE events again refer to the official docs for the same providers as in errors dialog

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first check which providers were covered in the errors dialog:


<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;await find_msgs(dname=&quot;/aai-ws/fastllm_v…&#x27;)→&quot;{&#x27;error&#x27;: &#x27;Failed to access aai-ws/fastl…&quot;</code></summary>

```json
{
  "id": "toolu_016wx5syiQRgjhA6cheTY5v8",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "await find_msgs(dname=\"/aai-ws/fastllm_v2/nbs/00_errors\", re_pattern=\"provider|Provider|official|docs\")"
    }
  },
  "result": "{'error': 'Failed to access aai-ws/fastllm_v2/nbs/00_errors in /Users/keremturgutlu. Does it exist?'}"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;find ~/aai-ws/fastllm_v2/nbs -name &#x27;*err…&quot;)→&quot;{&#x27;error&#x27;: DisallowedDest(&#x27;/dev/null&#x27;),\n …&quot;</code></summary>

```json
{
  "id": "toolu_011Lqy7eoD5ggvgA74iZ66yz",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "find ~/aai-ws/fastllm_v2/nbs -name '*error*' -o -name '*Error*' 2>/dev/null"
    }
  },
  "result": "{'error': DisallowedDest('/dev/null'),\n 'allowed_cmds': \"true; [; seq; git status; gunzip; basename; less; tree; cat; git add; find !{'-delete', '-execdir', '-okdir', '-exec', '-ok'}; yes; whereis; git show; egrep; tr; grep; sort; git cat-file; rg; diff; comm; printf; git stash list; cd; git rev-parse; nslookup; file; dirname; realpath; false; git log; git switch; test; git fetch; wc; date; dig; git describe; git tag; git config --get; tail; du; git commit; stat; curl; hostname; git remote; more; whoami; ping; ls; env; git checkout; unzip; unrar; export; git ls-tree; bunzip2; df; head; git blame; git shortlog; cal; git branch; column; ack; cmp; printenv; git ls-files; locate; wget; which; fgrep; git diff; bat; cut; uniq; uname; echo; host; ag; tar; pwd; type; uptime; git config --list\",\n 'allowed_dests': './; /tmp',\n 'suggestion': 'rerun using an allowed tool/dest, or ask user to provide permission'}"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;find ~/aai-ws/fastllm_v2/nbs -name &#x27;*err…&quot;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_01WEuoGRxRcwuxZLeZRJNofa",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "find ~/aai-ws/fastllm_v2/nbs -name '*error*' -o -name '*Error*'"
    }
  },
  "result": "---------------------------------------------------------------------------\nOSError                                   Traceback (most recent call last)\nCell In[28], line 1\n----> 1 await maybe_await(bash(cmd=\"find ~/aai-ws/fastllm_v2/nbs -name '*error*' -o -name '*Error*'\"))\n\nFile ~/aai-ws/safecmd/safecmd/core.py:326, in bash(cmd, as_dict, rm_cmds, rm_dests)\n    312 def bash(\n    313     cmd:str,  # Bash command string to execute - all shell features like pipes and subcommands are supported\n    314     as_dict:bool=False, # Return a dict response with 'success' or 'error' key\n    315     rm_cmds:str=None,  # Temp remove these commands from allow list\n    316     rm_dests:str=None  # Temp remove these destinations from allow list\n    317 ): # dict with 'success' or 'error' key; value is stdout+stderr for success, or error message otherwise\n    318     \"\"\"Run a bash shell command line safely and return the concatencated stdout and stderr.\n    319     Since it is run with `bash`, special chars like `$` and `*` are handled by the shell, so must be quoted if literal.\n    320     `cmd` is parsed and all calls are checked against an allow-list.\n   (...)    324     All operators are supported. Output redirects are validated against allowed destinations (default: ./ and /tmp).\n    325     rm_ params are comma-separated strs.\"\"\"\n--> 326     try: res = safe_run(cmd, rm_cmds=rm_cmds, rm_dests=rm_dests)\n    327     except PermissionError as e:\n    328         eff_cmds, eff_dests = _eff_sets(rm_cmds=rm_cmds, rm_dests=rm_dests)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:309, in safe_run(cmd, cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests, ignore_ex, split)\n    307 eff_cmds, eff_dests = _eff_sets(cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests)\n    308 validate(cmd, eff_cmds, eff_dests)\n--> 309 return run(cmd, ignore_ex=ignore_ex, split=split)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:32, in run(cmd, ignore_ex, split)\n     30 if res.stderr: out += res.stderr\n     31 if ignore_ex: return (res.returncode, out)\n---> 32 if res.returncode: raise IOError(out)\n     33 return out\n\nOSError: find: /Users/keremturgutlu/aai-ws/fastllm_v2/nbs: No such file or directory"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>bash(cmd=&#x27;ls ~/aai-ws/fastllm_v2/&#x27;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_01RT4h4H22Pg1qVHuntm8vea",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "ls ~/aai-ws/fastllm_v2/"
    }
  },
  "result": "---------------------------------------------------------------------------\nOSError                                   Traceback (most recent call last)\nCell In[29], line 1\n----> 1 await maybe_await(bash(cmd='ls ~/aai-ws/fastllm_v2/'))\n\nFile ~/aai-ws/safecmd/safecmd/core.py:326, in bash(cmd, as_dict, rm_cmds, rm_dests)\n    312 def bash(\n    313     cmd:str,  # Bash command string to execute - all shell features like pipes and subcommands are supported\n    314     as_dict:bool=False, # Return a dict response with 'success' or 'error' key\n    315     rm_cmds:str=None,  # Temp remove these commands from allow list\n    316     rm_dests:str=None  # Temp remove these destinations from allow list\n    317 ): # dict with 'success' or 'error' key; value is stdout+stderr for success, or error message otherwise\n    318     \"\"\"Run a bash shell command line safely and return the concatencated stdout and stderr.\n    319     Since it is run with `bash`, special chars like `$` and `*` are handled by the shell, so must be quoted if literal.\n    320     `cmd` is parsed and all calls are checked against an allow-list.\n   (...)    324     All operators are supported. Output redirects are validated against allowed destinations (default: ./ and /tmp).\n    325     rm_ params are comma-separated strs.\"\"\"\n--> 326     try: res = safe_run(cmd, rm_cmds=rm_cmds, rm_dests=rm_dests)\n    327     except PermissionError as e:\n    328         eff_cmds, eff_dests = _eff_sets(rm_cmds=rm_cmds, rm_dests=rm_dests)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:309, in safe_run(cmd, cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests, ignore_ex, split)\n    307 eff_cmds, eff_dests = _eff_sets(cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests)\n    308 validate(cmd, eff_cmds, eff_dests)\n--> 309 return run(cmd, ignore_ex=ignore_ex, split=split)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:32, in run(cmd, ignore_ex, split)\n     30 if res.stderr: out += res.stderr\n     31 if ignore_ex: return (res.returncode, out)\n---> 32 if res.returncode: raise IOError(out)\n     33 return out\n\nOSError: ls: /Users/keremturgutlu/aai-ws/fastllm_v2/: No such file or directory"
}
```

</details>

Looks like `fastllm_v2` doesn't exist on disk yet. Could you point me to the errors dialog path? Or if you have it open, what providers did you cover there — was it the usual suspects (OpenAI, Anthropic, Google)?

<details class='token-usage-details'><summary>$0.1977</summary>

`total=150,088 | in=149,608 | out=480 | cached=94.2% | cache_new=4,962 | $0.1977`

</details>


`aai-ws/fastllm/nbs/00_errors` - remember to make sequential web searches not in parallel

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check the errors dialog first:


<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;await find_msgs(dname=&quot;/aai-ws/fastllm/n…&#x27;)→&quot;[{&#x27;id&#x27;: &#x27;_b4e7c7ed&#x27;, &#x27;time_run&#x27;: &#x27;2026-0…&quot;</code></summary>

```json
{
  "id": "toolu_013WbLNVsCk8XkNPSmqJE9nF",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "await find_msgs(dname=\"/aai-ws/fastllm/nbs/00_errors\", re_pattern=\"provider|Provider|official|docs|http|anthropic|openai|google|gemini\")"
    }
  },
  "result": "<TRUNCATED>\u2026b4e7c7ed', 'time_run': '2026-03-31T09:59:15.125866+00:00', 'is_exported': 1, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': 'from fastcore.utils import *\\nimport json, httpx', 'output': '', 'msg_type': 'code', 'input_tokens': 53, 'output_tokens': {}}, {'id': '_2e68e8c6', 'time_run': '', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': {}, 'content': '## Purpose And Design\\n\\n`errors.py` defines the error contract used across the entire library.\\n\\n### Why this module exists\\n\\nProviders return different error shapes for HTTP and streaming failures. If each layer handled raw provider payloads directly, calling code would need provider-specific branches everywhere.\\n\\n### Architectural fit\\n\\n- Upstream inputs: raw `httpx` exceptions and SSE error events.\\n- Downstream consumers: `transport`, dynamic ops in `oapi`, provider `clients`, and high-level entrypoints.\\n\\n### Key design choices\\n\\n- `APIError` carries normalized fields (`status_code`, `error_type`, `code`, `request_id`, `retryable`, `raw`).\\n- Best-effort parsing keeps useful diagnostics without overfitting to one vendor.\\n- Context enrichment (`provider`, `model`, `endpoint`) helps debugging in multi-provider runs.\\n\\n### Reader outcome\\n\\nAfter this notebook, you should be able to trace where any failure is normalized and how error metadata remains consistent from low-level transport to user-facing calls.', 'output': '', 'msg_type': 'note', 'input_tokens': 323, 'output_tokens': {}}, {'id': '_fa3a0bc4', 'time_run': '', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': {}, 'content': '### `ProtocolError`\\n\\nRaised when provider payloads do not match expected protocol shape.\\n\\n**Role In This Module**\\n- Public symbol in this module API surface.\\n- Carries behavior central to this module stage of the pipeline.\\n\\n**Design Notes**\\n- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.\\n\\n**Connections**\\n- In-module dependencies used: primarily local logic.\\n- Referenced by modules: `normalize`, `transport`.\\n- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.', 'output': '', 'msg_type': 'note', 'input_tokens': 197, 'output_tokens': {}}, {'id': '_d5c03d80', 'time_run': '2026-03-31T09:59:15.265146+00:00', 'is_exported': 1, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': 'class ProtocolError(FastLLMError):\\n    \"Raised when provider payloads do not match expected protocol shape.\"', 'output': '', 'msg_type': 'code', 'input_tokens': 67, 'output_tokens': {}}, {'id': '_defaa637', 'time_run': '', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': {}, 'content': '### `_req_id`\\n\\nExtract provider request id header when available.\\n\\n**Role In This Module**\\n- Internal helper for module composition.\\n- Carries behavior central to this module stage of the pipeline.\\n\\n**Design Notes**\\n- Signature: `_req_id(headers: Optional[httpx.Headers])`\\n- Returns: `str`\\n- Small helper/function boundary keeps calling layers easier to compose and test.\\n\\n**Connections**\\n- In-module dependencies used: primarily local logic.\\n- Referenced by modules: none or only via orchestration.\\n- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.', 'output': '', 'msg_type': 'note', 'input_tokens': 215, 'output_tokens': {}}, {'id': '_9c1cdf65', 'time_run': '2026-03-31T09:59:15.374611+00:00', 'is_exported': 1, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': 'def _req_id(headers):\\n    \"Extract provider request id header when available.\"\\n    if headers is None: return \"\"\\n    for k in (\"x-request-id\", \"request-id\", \"anthropic-request-id\", \"x-goog-request-id\"):\\n        v = headers.get(k)\\n        if v: return str(v)\\n    return \"\"', 'output': '', 'msg_type': 'code', 'input_tokens': 130, 'output_tokens': {}}, {'id': '_87d33bba', 'time_run': '', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': {}, 'content': 'All three major providers (OpenAI, Anthropic, Gemini) consistently return `{\"error\": <dict>}` with a `message` field inside, so the primary `isinstance(err, dict)` branch handles them all. The remaining branches are defensive fallbacks f\u2026</TRUNCATED>"
}
```

</details>





So the errors dialog covers **OpenAI**, **Anthropic**, and **Gemini**. Now let me look up their official SSE streaming docs one at a time.



Now Anthropic:Now Gemini:Now let me also check the OpenAI Chat Completions SSE format (the legacy/common one used by fastllm), since the Responses API is newer:Great, I now have comprehensive info on all three providers. Here's a summary of how each provider uses SSE for streaming:

---

**OpenAI** — Chat Completions ([docs](https://developers.openai.com/api/docs/guides/streaming-responses))

[*](https://developers.openai.com/api/docs/guides/streaming-responses "Streaming API responses | OpenAI API") Set `stream=True` — this returns an object that streams back the response as **data-only** server-sent events. The SSE `event:` field is **not used** — everything goes through `data:`. Each chunk is a `ChatCompletionChunk` JSON object with `choices[0].delta.content` for text. [*](https://github.com/llamastack/llama-stack/issues/4744 "SDK streaming fails on OpenAI-style data:[DONE] terminator for chat completions (TypeScript/Go/Python) · Issue #4744 · llamastack/llama-stack") The stream ends with a final `data:[DONE]` chunk. Error events during streaming use the flat `{code, message}` shape (as noted in the errors dialog).

**Anthropic** — Messages ([docs](https://docs.anthropic.com/en/api/messages-streaming))

[*](https://docs.anthropic.com/en/api/messages-streaming?debug_url=1&debug=1&debug=true "Streaming Messages - Anthropic") Each server-sent event includes a named event type and associated JSON data. Each event uses an SSE `event` name (e.g. `event: message_stop`), and includes the matching event type in its data. The lifecycle is: [*](https://docs.anthropic.com/en/api/messages-streaming?debug_url=1&debug=1&debug=true "Streaming Messages - Anthropic") `content_block_start` → one or more `content_block_delta` events → `content_block_stop`, plus `message_delta` events for top-level changes, a final `message_stop` event, and ping events throughout. [*](https://docs.anthropic.com/claude/reference/versions "Versions - Claude - Anthropic") All events are named events, rather than data-only events. No `data: [DONE]` event. [*](https://docs.anthropic.com/en/api/messages-streaming?debug_url=1&debug=1&debug=true "Streaming Messages - Anthropic") Errors may be sent in the event stream, e.g. `event: error` with `data: {"type": "error", "error": {"type": "overloaded_error", "message": "Overloaded"}}`.

**Gemini** — streamGenerateContent ([docs](https://ai.google.dev/api/generate-content))

[*](https://ai.google.dev/api "Gemini API reference | Google AI for Developers") Uses Server-Sent Events (SSE) to push chunks of the response as they are generated. [*](https://github.com/google-gemini/cookbook/blob/main/quickstarts/rest/Streaming_REST.ipynb "cookbook/quickstarts/rest/Streaming_REST.ipynb at main · google-gemini/cookbook") You must set `alt=sse` in the URL. Each stream chunk is a `GenerateContentResponse` object with a portion of the output text in `candidates[0].content.parts[0].text`. Like OpenAI, Gemini uses **data-only** events (no named `event:` field). [*](https://ai.google.dev/api "Gemini API reference | Google AI for Developers") Each response contains a `responseId` that ties the full response together. Errors use the same `{error: {code, message, status, details}}` shape as non-streaming responses.

**Key differences for `SSEvent` parsing:**

| | SSE `event:` field | `data:` format | Stream terminator | Error shape |
|---|---|---|---|---|
| **OpenAI** | Not used (data-only) | `ChatCompletionChunk` JSON | `data: [DONE]` | Flat `{code, message}` |
| **Anthropic** | Named events (`message_start`, `content_block_delta`, etc.) | JSON with `type` field matching `event:` | `event: message_stop` | `{type: "error", error: {type, message}}` |
| **Gemini** | Not used (data-only) | `GenerateContentResponse` JSON | Stream ends naturally | `{error: {code, message, status}}` |

This is directly relevant to `SSEvent` — Anthropic is the only provider that uses the SSE `event` field, while OpenAI and Gemini are data-only. The `id` and `retry` SSE fields aren't used by any of these providers in practice.

<details class='token-usage-details'><summary>$1.0541</summary>

`total=374,665 | in=373,040 | out=1,625 | cached=77.2% | cache_new=74,098 | searches=4 | $1.0541`

</details>


### `parse_sse_lines`

Parses synchronous SSE line streams into `SSEvent` objects.

**Why it exists**
- Provides a minimal framing parser reusable in tests and non-async contexts.

**Connections**
- Conceptual counterpart to `aiter_sse` used in async transport paths.

Ok let's send api calls for each provider and parse few lines

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great idea! Let's start with one provider at a time so we can inspect the raw SSE lines before parsing.

Which provider do you want to hit first — **OpenAI**, **Anthropic**, or **Gemini**? The basic approach for each will be:

1. Make a streaming `httpx` POST request with `stream=True`
2. Collect the raw SSE lines
3. Feed them into `parse_sse_lines` and inspect the resulting `SSEvent` objects

I'd suggest starting with **Anthropic** since it's the most interesting case (it actually uses the `event:` field), but up to you!

<details class='token-usage-details'><summary>$0.0672</summary>

`total=35,090 | in=34,845 | out=245 | cached=89.0% | cache_new=3,208 | $0.0672`

</details>


Write examples for each similar to how we did in errors dialog

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check how the examples were structured in the errors dialog first:


<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;res = await find_msgs(dname=&quot;/aai-ws/fas…&#x27;)→&#x27;--- _b4e7c7ed (code) ---\nfrom fastcore.u…&#x27;</code></summary>

```json
{
  "id": "toolu_01Lim9yLeyZyM9eSejXwRUKy",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "res = await find_msgs(dname=\"/aai-ws/fastllm/nbs/00_errors\", re_pattern=\"httpx|test_eq|Example\")\nfor m in res:\n    if m.get('content','').strip():\n        print(f\"--- {m['id']} ({m['msg_type']}) ---\")\n        print(m['content'][:300])\n        print()"
    }
  },
  "result": "--- _b4e7c7ed (code) ---\nfrom fastcore.utils import *\nimport json, httpx\n\n--- _2e68e8c6 (note) ---\n## Purpose And Design\n\n`errors.py` defines the error contract used across the entire library.\n\n### Why this module exists\n\nProviders return different error shapes for HTTP and streaming failures. If each layer handled raw provider payloads directly, calling code would need provider-specific branches\n\n--- _defaa637 (note) ---\n### `_req_id`\n\nExtract provider request id header when available.\n\n**Role In This Module**\n- Internal helper for module composition.\n- Carries behavior central to this module stage of the pipeline.\n\n**Design Notes**\n- Signature: `_req_id(headers: Optional[httpx.Headers])`\n- Returns: `str`\n- Small he\n\n--- _fdf25eba (code) ---\n@patch\ndef error(self: httpx.Response):\n    \"Parse common HTTP API error shapes to (message, error_type, code, raw).\"\n    try: raw = self.json()\n    except Exception: raw = self.text\n\n    msg, et, code, status = \"\", \"\", None, None\n    if isinstance(raw, dict):\n        err = raw.get(\"error\")\n        \n\n--- _4b8a628f (code) ---\nresp = httpx.post(\n    \"https://api.openai.com/v1/chat/completions\",\n    headers={\"Authorization\": \"Bearer sk-fake123\"},\n    json={\"model\": \"gpt-4o\", \"messages\": [{\"role\": \"user\", \"content\": \"hi\"}]}\n)\nresp.json()\n\n--- _69322fb9 (code) ---\nresp = httpx.post(\n    \"https://api.anthropic.com/v1/messages\",\n    headers={\"x-api-key\": \"sk-fake123\", \"anthropic-version\": \"2023-06-01\"},\n    json={\"model\": \"claude-sonnet-4-20250514\", \"max_tokens\": 10, \"messages\": [{\"role\": \"user\", \"content\": \"hi\"}]}\n)\nresp.json()\n\n--- _c7dd4616 (code) ---\nresp = httpx.post(\n    \"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key=fake123\",\n    json={\"contents\": [{\"parts\": [{\"text\": \"hi\"}]}]}\n)\nresp.json()\n\n--- _6bf75d33 (prompt) ---\nCan you walk me through this function line by line having all 3 provider example error responses, annotate each line why each needed. Let's try to see if it can be simplified\n\n--- _cde0bc9b (prompt) ---\nDo SSE events have a type like httpx.Response like we can patch?\n\n--- _1ad7307c (code) ---\nresp = httpx.post(\n    \"https://api.anthropic.com/v1/messages\",\n    headers={\"x-api-key\": \"sk-fake123\", \"anthropic-version\": \"2023-06-01\"},\n    json={\"model\": \"claude-sonnet-4-20250514\", \"max_tokens\": 10, \"messages\": [{\"role\": \"user\", \"content\": \"hi\"}]}\n)\ne = resp.error()\nerr = APIError(e.message, p\n\n--- _b5140f19 (code) ---\nresp = httpx.post(\n    \"https://api.anthropic.com/v1/messages\",\n    headers={\"x-api-key\": \"sk-fake123\", \"anthropic-version\": \"2023-06-01\"},\n    json={\"model\": \"claude-sonnet-4-20250514\", \"max_tokens\": 10, \"messages\": [{\"role\": \"user\", \"content\": \"hello!\"}]}\n)\ne = resp.error()\nerr = APIError(e.messag\n\n--- _9cfc6ee4 (note) ---\n### `api_error_from_http`\n\nConverts `httpx.HTTPStatusError` into `APIError`.\n\n**Why it exists**\n- HTTP error payloads differ by provider and endpoint.\n\n**Design Notes**\n- Parses best-effort structured fields and computes retryability hints.\n\n**Connections**\n- Core adapter used by transport/oapi/clie\n\n--- _83d2dc0c (code) ---\n@patch\ndef api_error(self:httpx.HTTPStatusError, *, provider: str = \"\", model: str = \"\", endpoint: str = \"\"):\n    \"Build APIError from httpx HTTPStatusError.\"\n    resp = self.response\n    err = resp.error()\n    if not endpoint and resp.request is not None:\n        endpoint = f\"{resp.request.method.u\n\n--- _90778e19 (code) ---\nresp = httpx.post(\n    \"https://api.anthropic.com/v1/messages\",\n    headers={\"x-api-key\": \"sk-fake123\", \"anthropic-version\": \"2023-06-01\"},\n    json={\"model\": \"claude-sonnet-4-20250514\", \"max_tokens\": 10, \"messages\": [{\"role\": \"user\", \"content\": \"test api_error_from_http\"}]}\n)\ntry: resp.raise_for_st\n\n--- _8039df71 (prompt) ---\nI meant the example tests above in this dialog, are they properly showing non-200?\n\n--- _5876eda9 (prompt) ---\nDoes it make sense to make `api_error_from_http` a patch on `httpx.HTTPStatusError`?"
}
```

</details>

I can see the pattern — simple `httpx.post` calls showing real API responses for each provider. For SSE streaming, we'll do the same but with `stream=True` and real API keys. Here's the approach for all three:

```python
# OpenAI streaming
import os
with httpx.stream("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json={"model": "gpt-4o-mini", "stream": True, "max_tokens": 20,
          "messages": [{"role": "user", "content": "Say hello"}]}
) as r:
    oai_lines = list(r.iter_lines())
oai_lines[:10]
```

```python
# Anthropic streaming
with httpx.stream("POST", "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "stream": True, "max_tokens": 20,
          "messages": [{"role": "user", "content": "Say hello"}]}
) as r:
    ant_lines = list(r.iter_lines())
ant_lines[:10]
```

```python
# Gemini streaming
with httpx.stream("POST",
    f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse&key={os.environ['GOOGLE_API_KEY']}",
    json={"contents": [{"parts": [{"text": "Say hello"}]}]}
) as r:
    gem_lines = list(r.iter_lines())
gem_lines[:10]
```

Try running these one at a time — we'll inspect the raw lines first, then feed them through `parse_sse_lines` to see the parsed `SSEvent` objects for each provider.

<details class='token-usage-details'><summary>$0.0958</summary>

`total=72,455 | in=71,713 | out=742 | cached=97.2% | cache_new=220 | $0.0958`

</details>


This is not exported as it's not needed since we keep the higher level api async only for simplicity:

In [ ]:
def parse_sse_lines(lines):
    "Parse SSE line stream into events."
    event, data, event_id, retry = None, [], None, None
    for raw in lines:
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)

In [ ]:
with httpx.stream("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json={"model": "gpt-4o-mini", "stream": True, "max_tokens": 20,
          "messages": [{"role": "user", "content": "Say hello"}]}
) as r:
    oai_lines = list(r.iter_lines())
oai_lines[:10]

['data: {"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"role":"assistant","content":"","refusal":null},"logprobs":null,"finish_reason":null}],"obfuscation":"k0Usqv"}',
 '',
 'data: {"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"Hello"},"logprobs":null,"finish_reason":null}],"obfuscation":"vBg"}',
 '',
 'data: {"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"!"},"logprobs":null,"finish_reason":null}],"obfuscation":"DlpNx9E"}',
 '',
 'dat

In [ ]:
with httpx.stream("POST", "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "stream": True, "max_tokens": 20,
          "messages": [{"role": "user", "content": "Say hello"}]}
) as r:
    ant_lines = list(r.iter_lines())
ant_lines[:10]

['event: message_start',
 'data: {"type":"message_start","message":{"model":"claude-sonnet-4-20250514","id":"msg_015ZCvz5Gqn8CHzRub5DfZJ8","type":"message","role":"assistant","content":[],"stop_reason":null,"stop_sequence":null,"usage":{"input_tokens":9,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}} }',
 '',
 'event: content_block_start',
 'data: {"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}               }',
 '',
 'event: ping',
 'data: {"type": "ping"}',
 '',
 'event: content_block_delta']

In [ ]:
with httpx.stream("POST",
    f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:streamGenerateContent?alt=sse&key={os.environ['GEMINI_API_KEY']}",
    json={"contents": [{"parts": [{"text": "Say hello"}]}]}
) as r:
    gem_lines = list(r.iter_lines())
gem_lines[:10]

['data: {"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}',
 '']

In [ ]:
#| export
@patch
def data_json(self: SSEvent): 
    try: return json.loads(self.data)
    except: return self.data

@patch
def data_obj(self: SSEvent): return dict2obj(self.data_json())

In [ ]:
oai_sse = L(parse_sse_lines(oai_lines))
ant_sse = L(parse_sse_lines(ant_lines))
gem_sse = L(parse_sse_lines(gem_lines))

In [ ]:
oai_sse[:3]

[SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"role":"assistant","content":"","refusal":null},"logprobs":null,"finish_reason":null}],"obfuscation":"k0Usqv"}', id=None, retry=None), SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"Hello"},"logprobs":null,"finish_reason":null}],"obfuscation":"vBg"}', id=None, retry=None), SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"c

In [ ]:
oai_sse[-1]

SSEvent(event=None, data='[DONE]', id=None, retry=None)

In [ ]:
oai_sse[0].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': '', 'refusal': {}}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'k0Usqv',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse[1].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'content': 'Hello'}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'vBg',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse[-1].data_obj()

'[DONE]'

In [ ]:
ant_sse[:3]

[SSEvent(event='message_start', data='{"type":"message_start","message":{"model":"claude-sonnet-4-20250514","id":"msg_015ZCvz5Gqn8CHzRub5DfZJ8","type":"message","role":"assistant","content":[],"stop_reason":null,"stop_sequence":null,"usage":{"input_tokens":9,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}} }', id=None, retry=None), SSEvent(event='content_block_start', data='{"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}               }', id=None, retry=None), SSEvent(event='ping', data='{"type": "ping"}', id=None, retry=None)]

In [ ]:
ant_sse[-1]

SSEvent(event='message_stop', data='{"type":"message_stop"          }', id=None, retry=None)

In [ ]:
ant_sse[0].data_obj()

```python
{ 'message': { 'content': [],
               'id': 'msg_015ZCvz5Gqn8CHzRub5DfZJ8',
               'model': 'claude-sonnet-4-20250514',
               'role': 'assistant',
               'stop_reason': {},
               'stop_sequence': {},
               'type': 'message',
               'usage': { 'cache_creation': { 'ephemeral_1h_input_tokens': 0,
                                              'ephemeral_5m_input_tokens': 0},
                          'cache_creation_input_tokens': 0,
                          'cache_read_input_tokens': 0,
                          'inference_geo': 'not_available',
                          'input_tokens': 9,
                          'output_tokens': 4,
                          'service_tier': 'standard'}},
  'type': 'message_start'}
```

In [ ]:
ant_sse[1].data_obj()

```python
{ 'content_block': {'text': '', 'type': 'text'},
  'index': 0,
  'type': 'content_block_start'}
```

In [ ]:
ant_sse[2].data_obj()

```python
{'type': 'ping'}
```

In [ ]:
ant_sse[3].data_obj()

```python
{ 'delta': {'text': 'Hello! How are', 'type': 'text_delta'},
  'index': 0,
  'type': 'content_block_delta'}
```

In [ ]:
ant_sse[-1].data_obj()

```python
{'type': 'message_stop'}
```

In [ ]:
gem_sse[:3]

[SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)]

In [ ]:
gem_sse[-1]

SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)

In [ ]:
gem_sse[0].data_obj()

```python
{ 'candidates': [{'content': {'parts': [{'text': 'Hello!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}],
  'modelVersion': 'gemini-2.5-flash',
  'responseId': 'E6bLabD3FqvujrEP1fel0A4',
  'usageMetadata': { 'candidatesTokenCount': 2,
                     'promptTokenCount': 2,
                     'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2}],
                     'thoughtsTokenCount': 15,
                     'totalTokenCount': 19}}
```

### `aiter_sse`

Async SSE parser over an `httpx` streamed response.

**Why it exists**
- Streaming providers emit event frames incrementally; this parser isolates framing from provider semantics.

**Connections**
- Called by `transport.stream_sse`, then decoded/normalized downstream.

In [ ]:
#| export
async def aiter_sse(response: httpx.Response):
    "Async SSE parser from an httpx streamed response."
    event, data, event_id, retry = None, [], None, None
    async for raw in response.aiter_lines():
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)

Why do we have both `aiter_sse` and `parse_sse_lines`?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

They share identical parsing logic but serve different input sources:

- **`parse_sse_lines`** — takes a synchronous iterable of strings (like a `list`). Great for tests, debugging, and non-async contexts — as you just did above with `list(r.iter_lines())`.

- **`aiter_sse`** — takes a live `httpx.Response` and uses `response.aiter_lines()` to parse incrementally as bytes arrive over the wire. This is what the real streaming transport uses, since you want to yield `SSEvent`s as they arrive rather than buffering everything first.

The key difference is **when** data is available: `aiter_sse` can yield events the instant each one finishes arriving, while `parse_sse_lines` needs all lines upfront. For an LLM streaming response, that latency difference matters — you want tokens on screen as fast as possible.

<details class='token-usage-details'><summary>$0.3957</summary>

`total=42,588 | in=42,340 | out=248 | cached=0.0% | cache_new=36,727 | $0.3957`

</details>


Can you write the same examples we hava above in `parse_sse_lines` section for `aiter_sse`

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! Here are the async equivalents using `aiter_sse` directly on the streaming response:

```python
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST", "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        json={"model": "gpt-4o-mini", "stream": True, "max_tokens": 20,
              "messages": [{"role": "user", "content": "Say hello"}]}
    ) as r:
        oai_sse2 = [e async for e in aiter_sse(r)]
oai_sse2[:3]
```

```python
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST", "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"},
        json={"model": "claude-sonnet-4-20250514", "stream": True, "max_tokens": 20,
              "messages": [{"role": "user", "content": "Say hello"}]}
    ) as r:
        ant_sse2 = [e async for e in aiter_sse(r)]
ant_sse2[:3]
```

```python
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST",
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:streamGenerateContent?alt=sse&key={os.environ['GEMINI_API_KEY']}",
        json={"contents": [{"parts": [{"text": "Say hello"}]}]}
    ) as r:
        gem_sse2 = [e async for e in aiter_sse(r)]
gem_sse2[:3]
```

Same pattern as the sync versions — the only difference is `httpx.AsyncClient` + `cli.stream()` context manager, and passing the response directly to `aiter_sse` instead of collecting lines first. The resulting `SSEvent` objects should be identical.

<details class='token-usage-details'><summary>$0.2388</summary>

`total=43,324 | in=42,635 | out=689 | cached=50.7% | cache_new=20,801 | $0.2388`

</details>


In [ ]:
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST", "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        json={"model": "gpt-4o-mini", "stream": True, "max_tokens": 20,
              "messages": [{"role": "user", "content": "Say hello"}]}
    ) as r:
        oai_sse2 = [e async for e in aiter_sse(r)]
oai_sse2[:3]

[SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"role":"assistant","content":"","refusal":null},"logprobs":null,"finish_reason":null}],"obfuscation":"k0Usqv"}', id=None, retry=None),
 SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"Hello"},"logprobs":null,"finish_reason":null}],"obfuscation":"vBg"}', id=None, retry=None),
 SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{

In [ ]:
oai_sse2[-1]

SSEvent(event=None, data='[DONE]', id=None, retry=None)

In [ ]:
oai_sse2[0].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': '', 'refusal': {}}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'k0Usqv',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse2[1].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'content': 'Hello'}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'vBg',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse2[-1].data_obj()

'[DONE]'

In [ ]:
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST", "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"},
        json={"model": "claude-sonnet-4-20250514", "stream": True, "max_tokens": 20,
              "messages": [{"role": "user", "content": "Say hello"}]}
    ) as r:
        ant_sse2 = [e async for e in aiter_sse(r)]
ant_sse2[:3]

[SSEvent(event='message_start', data='{"type":"message_start","message":{"model":"claude-sonnet-4-20250514","id":"msg_015ZCvz5Gqn8CHzRub5DfZJ8","type":"message","role":"assistant","content":[],"stop_reason":null,"stop_sequence":null,"usage":{"input_tokens":9,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}} }', id=None, retry=None),
 SSEvent(event='content_block_start', data='{"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}               }', id=None, retry=None),
 SSEvent(event='ping', data='{"type": "ping"}', id=None, retry=None)]

In [ ]:
ant_sse2[-1]

SSEvent(event='message_stop', data='{"type":"message_stop"          }', id=None, retry=None)

In [ ]:
ant_sse2[0].data_obj()

```python
{ 'message': { 'content': [],
               'id': 'msg_015ZCvz5Gqn8CHzRub5DfZJ8',
               'model': 'claude-sonnet-4-20250514',
               'role': 'assistant',
               'stop_reason': {},
               'stop_sequence': {},
               'type': 'message',
               'usage': { 'cache_creation': { 'ephemeral_1h_input_tokens': 0,
                                              'ephemeral_5m_input_tokens': 0},
                          'cache_creation_input_tokens': 0,
                          'cache_read_input_tokens': 0,
                          'inference_geo': 'not_available',
                          'input_tokens': 9,
                          'output_tokens': 4,
                          'service_tier': 'standard'}},
  'type': 'message_start'}
```

In [ ]:
ant_sse2[1].data_obj()

```python
{ 'content_block': {'text': '', 'type': 'text'},
  'index': 0,
  'type': 'content_block_start'}
```

In [ ]:
ant_sse2[2].data_obj()

```python
{'type': 'ping'}
```

In [ ]:
ant_sse2[3].data_obj()

```python
{ 'delta': {'text': 'Hello! How are', 'type': 'text_delta'},
  'index': 0,
  'type': 'content_block_delta'}
```

In [ ]:
ant_sse2[-1].data_obj()

```python
{'type': 'message_stop'}
```

In [ ]:
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST",
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:streamGenerateContent?alt=sse&key={os.environ['GEMINI_API_KEY']}",
        json={"contents": [{"parts": [{"text": "Say hello"}]}]}
    ) as r:
        gem_sse2 = [e async for e in aiter_sse(r)]
gem_sse2[:3]

[SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)]

In [ ]:
gem_sse2[-1]

SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)

In [ ]:
gem_sse2[0].data_obj()

```python
{ 'candidates': [{'content': {'parts': [{'text': 'Hello!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}],
  'modelVersion': 'gemini-2.5-flash',
  'responseId': 'E6bLabD3FqvujrEP1fel0A4',
  'usageMetadata': { 'candidatesTokenCount': 2,
                     'promptTokenCount': 2,
                     'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2}],
                     'thoughtsTokenCount': 15,
                     'totalTokenCount': 19}}
```

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()